# 06 Tukey HSD

ANOVA後に、どのグループ同士が違うかを確認するNotebook。

本来はANOVAで有意差が出た場合に使うが、ここでは参考値として有意差が出ていなくても実行できる。


In [ ]:
from pathlib import Path

import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd

def find_project_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data').exists() and (candidate / 'はるた').exists():
            return candidate
    raise FileNotFoundError('プロジェクトルートが見つかりません。data/ と はるた/ がある場所で実行してください。')


PROJECT_ROOT = find_project_root()
HARUTA_ROOT = PROJECT_ROOT / 'はるた'
MATCH_DATE = '2023-11-25'
MATCH_ID = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / DATASET_NAME
TABLES_DIR = HARUTA_ROOT / 'outputs' / 'tables' / DATASET_NAME
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / 'team_centroids_by_sequence.csv'
ANOVA_PATH = TABLES_DIR / 'anova.csv'
OUTPUT_PATH = TABLES_DIR / 'tukey_hsd.csv'

ALPHA = 0.05
VALUE_COLS = ['mean_centroid_x', 'mean_centroid_y']
REQUIRED_COLUMNS = ['group', *VALUE_COLS]

DATASET_NAME, INPUT_PATH, ANOVA_PATH, OUTPUT_PATH


In [ ]:
def load_centroid_sequences(path):
    if not path.exists():
        raise FileNotFoundError(f'重心分析用CSVが見つかりません: {path}')
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'必須列が不足しています: {missing}')
    if df.empty:
        raise ValueError('入力CSVが0行です。先に 00_data_check.ipynb を再実行してください。')
    return df


def load_anova_results(path):
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def run_tukey_for_variable(df, value_col, alpha=ALPHA):
    valid = df[['group', value_col]].dropna()
    if valid['group'].nunique() < 2:
        raise ValueError(f'{value_col}: Tukey HSDには2グループ以上必要です。')

    tukey = pairwise_tukeyhsd(
        endog=valid[value_col],
        groups=valid['group'],
        alpha=alpha,
    )
    result = pd.DataFrame(
        tukey.summary().data[1:],
        columns=tukey.summary().data[0],
    )
    result.insert(0, 'variable', value_col)
    result.insert(0, 'test', 'tukey_hsd')
    return result


def run_tukey_tests(df, value_cols=VALUE_COLS):
    return pd.concat(
        [run_tukey_for_variable(df, col) for col in value_cols],
        ignore_index=True,
    )


## 入力確認


In [ ]:
df = load_centroid_sequences(INPUT_PATH)
anova = load_anova_results(ANOVA_PATH)

display(df.head())
if not anova.empty:
    display(anova[['variable', 'p_value', 'significant', 'note']])


## Tukey HSDを実行

ANOVAが有意でなくても、参考値として全変数で実行する。


In [ ]:
tukey_results = run_tukey_tests(df)
tukey_results


## 保存


In [ ]:
tukey_results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH}')
